# Import

In [ ]:
import colorsys
from itertools import combinations, product, permutations
import json
import matplotlib
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.ticker import FuncFormatter
from matplotlib import (pylab, rc, transforms)
import numpy as np
import openturns as ot
import os
import pandas as pd
from pprint import pprint
import random
import requests
import scipy
from scipy import (stats, linalg, special, integrate)
from scipy.integrate import (trapezoid, cumulative_trapezoid)
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.signal import argrelextrema
from scipy.stats import (anderson, energy_distance, gaussian_kde, kendalltau, ks_2samp, 
                         lognorm, norm, pearsonr, rv_continuous, spearmanr, wasserstein_distance)
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
from sklearn.preprocessing import LabelEncoder
import string
import time
from tqdm import tqdm
from typing import Optional
import warnings

matplotlib.rcParams['figure.dpi'] = 1200
alphabet = string.ascii_lowercase

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../../shared')

from funcs_unit_conversion import hi2, hi3, lo2

from customstats import weighted_lognorm_fit, shapiro_wilk_weighted, _royston_pvalue, empirical_metadata, NestedDictValues, weighted_ecdf, estimate_maxima, weighted_kurtosis, weighted_skew, wasserstein1_weighted, wasserstein2_weighted, weighted_mean, weighted_var, weighted_distance_norm, weighted_quantile, weighted_bw, weighted_std

from datageneration import (random_samples, generate_random_numbers, random_irregular_dataset, random_logcount)

from datavisualization import scale_lightness

from funcs_unit_conversion import dict_unitconv, dict_unittype, consistent_units, str2valunit


In [ ]:
# read in dictionary with labels for each metric
with open('../src/dct_metriclabels.json', 'r') as file:
    dct_metriclabels = json.load(file)


In [ ]:
# create dictionary with "1st" for integers "1"
rankth = {}

for i in np.arange(1,100):
    if i % 10 == 1:
        rankth[i] = f'{i}st'
    elif i % 10 == 2:
        rankth[i] = f'{i}nd'
    elif i % 10 == 3:
        rankth[i] = f'{i}rd'
    else:
        rankth[i] = f'{i}th'
rankth[11] = '11th'
rankth[12] = '12th'
rankth[13] = '13th'

# Import data and calculate statistical metrics of datasets

In [ ]:
# read data from file
with open('../data/processed/DATA_all.json', 'r') as file:
    DATA_all = json.load(file)

# convert lists to arrays
for dataset in DATA_all:
    DATA_all[dataset]['data'] = np.array(DATA_all[dataset]['data'])
    DATA_all[dataset]['weights'] = np.array(DATA_all[dataset]['weights'])

# read lists that trim data
with open('../data/processed/datasets_outliers.json', 'r') as f:
    datasets_outliers = json.load(f)
with open('../data/processed/datasets_trimto10k.json', 'r') as f:
    datasets_trimto10k = json.load(f)

# create new, trimmed dataset
datasets = DATA_all.keys()
datasets_trimmed = [ele for ele in datasets if ele not in datasets_trimto10k and ele not in datasets_outliers]
DATA = {key: DATA_all[key] for key in datasets_trimmed}

In [ ]:
# put all metrics into a single dataframe
dataset = list(DATA.keys())[0]
df_metrics = pd.DataFrame(columns=DATA[dataset]['metrics'].keys())
for dataset in tqdm(DATA, total=len(DATA)):
    df_metrics.loc[dataset] = DATA[dataset]['metrics']



# Fit probabilistic models

In [ ]:
# fit probabilistic models to each distribution (KDE, Normal, Lognormal) + (Uniform, Variable, Dirichlet)
# create ECC models - new version with three probability estimation methods (pe) and three weighting methods (wt)
# df_ecc = pd.DataFrame()
# nsamples = 10_000
prob_models = {k:{} for k in DATA.keys()}
PE = ['Normal', 'Lognormal', 'KDE']
WT = ['Uniform', 'Variable']

PEWT = []
for pe, wt in product(PE, WT):
    PEWT.append(f'{pe}, {wt}')

dct_colors = {pewt: color for pewt, color in zip(PEWT, sns.color_palette('Paired', len(PEWT)))}


logfit_offset = 0.5


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for dataset in tqdm(DATA, total=len(DATA)):
        #wt 1 - uniform weights
        #wt 2 - variable weights

        X = DATA[dataset]['data']
        W_uni = np.ones_like(X)/len(X)
        W_var = DATA[dataset]['weights']
        
        for W, wt in zip([W_uni, W_var], ['Uniform', 'Variable']):

            #Normal
            pe = 'Normal'
            pewt = f'{pe}, {wt}'
            mean = np.sum(X*W)/np.sum(W)
            std = weighted_std(X, W)
            func = norm(loc=mean, scale=std)
            prob_models[dataset][pewt] = func

            #Lognormal
            pe = 'Lognormal'
            pewt = f'{pe}, {wt}'
            shape, loc, scale = weighted_lognorm_fit(X+logfit_offset, weights=W, method='MLE')
            func = lognorm(s=shape, loc=loc-logfit_offset, scale=scale)
            prob_models[dataset][pewt] = func

            #KDE
            pe = 'KDE'
            pewt = f'{pe}, {wt}'
            bw = weighted_bw(X, W, bw_method='scott')
            func = gaussian_kde(X, bw_method=1.0, weights=W)
            bw_base = (func.covariance**0.5)[0][0]
            func.set_bandwidth(bw/bw_base)
            prob_models[dataset][pewt] = func

In [ ]:
# visualize each method
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
weights = np.random.dirichlet(np.ones_like(X))

PE = ['Normal', 'Lognormal', 'KDE']
WT = ['Uniform', 'Variable']

xplot = np.linspace(0,30,1_000)

fig, axes = plt.subplots(len(WT), len(PE), figsize=(8,4))
for col, pe in enumerate(PE):
    for row, wt in enumerate(WT):
        ax = axes[row, col]

        # set weights
        if wt == 'Uniform':
            W = np.ones_like(X)/len(X)
        else:
            W = weights.copy()

        # create function
        if pe == 'Normal':
            mean = np.sum(X*W)/np.sum(W)
            std = weighted_std(X, W)
            func = norm(loc=mean, scale=std)
        
        elif pe == 'Lognormal':
            shape, loc, scale = weighted_lognorm_fit(X+logfit_offset, weights=W, method='MLE')
            func = lognorm(s=shape, loc=loc-logfit_offset, scale=scale)

        elif pe == 'KDE':
            bw = weighted_bw(X, W, bw_method='scott')
            func = gaussian_kde(X, bw_method=1.0, weights=W)
            bw_base = (func.covariance**0.5)[0][0]
            func.set_bandwidth(bw/bw_base)
            ax.plot(xplot, (norm.pdf(xplot.reshape(-1,1), X, bw)*W), color='black', linewidth=0.5)

        # plot
        pewt = f'{pe}, {wt}'
        yplot = func.pdf(xplot)
        ax.plot(xplot, yplot, color=dct_colors[pewt])
            

        # add data
        ax.scatter(X, [0]*len(X), marker='|', s=W*1000, color='black')

        # add labels
        if row == 0:
            ax.text(0.5, 1.1, pe, ha='center', va='bottom', fontsize=12, transform=ax.transAxes)
        if col == 0:
            ax.text(-0.05, 0.5, f'{wt}\nWeighting', ha='right', va='center', fontsize=12, transform=ax.transAxes)

        # format
        ax.spines[['top', 'right' ,'left']].set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.set_xticks([])
        ax.set_ylim(0,)


In [ ]:
# show what wasserstein distance looks like
data = np.random.uniform(5,15,20)
weights = np.random.dirichlet(np.ones_like(data))
xplot = np.linspace(0,20,1_000)

# calculate function for ecdf
func_ecdf = weighted_ecdf(data, weights)[2]

# calculate function for probabilistic model
bw = weighted_bw(data, weights, bw_method='scott')
func = gaussian_kde(data, bw_method=1.0, weights=weights)
bw_base = (func.covariance**0.5)[0][0]
func.set_bandwidth(bw/bw_base)
modelpdf = func.pdf(xplot)
modelcdf = cumulative_trapezoid(modelpdf, xplot, initial=0)

fig, axes = plt.subplots(2,1, figsize=(5,3))

# PDF
ax = axes[0]
ax.plot(xplot, modelpdf, color='tab:blue', label='Probabilistic Model')
ax.scatter(data, [0]*len(data), color='black', label='Dataset', s=weights*2_000, marker='|')
ax.text(0,1,'PDF',ha='left', va='top', transform=ax.transAxes)


# CDF
ax = axes[1]
ax.plot(xplot, modelcdf, color='tab:blue', label='Probabilistic Model')
ax.plot(xplot, func_ecdf(xplot), color='black', label='Dataset')
ax.text(0,1,'CDF',ha='left', va='top', transform=ax.transAxes)
ax.fill_between(xplot, modelcdf, func_ecdf(xplot), alpha=0.5, color='tab:green', label='Wasserstein-1 Distance')
ax.legend(bbox_to_anchor=(0.6,0.05), loc='lower left', fontsize=8)

# format
for i in range(2):
    ax = axes[i]
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_ylim(0,)
    ax.set_xticks([])

plt.savefig('../outputs/figures/CompareUQMethods_DEF_DemoW1Dist.png', bbox_inches='tight')


# Pull in empirical data and fit probabilistic models

In [ ]:
with open('../data/processed/dct_realeccs_trimmed.json', 'r') as file:
    dct_realeccs_trimmed = json.load(file)

for mat in dct_realeccs_trimmed:
    dct_realeccs_trimmed[mat]['data'] = np.array(dct_realeccs_trimmed[mat]['data'])
    dct_realeccs_trimmed[mat]['weights'] = np.array(dct_realeccs_trimmed[mat]['weights'])

df_realeccmetrics = pd.read_excel('../outputs/tables/TABLE_EmpiricalECCMetrics.xlsx', index_col=0)


In [ ]:
# apply each method to empirical ECCs and compare their fits

# fit probabilistic models to each distribution (KDE, Normal, Lognormal) + (Uniform, Variable, Dirichlet)
# create ECC models - new version with three probability estimation methods (pe) and three weighting methods (wt)
# df_ecc = pd.DataFrame()
# nsamples = 10_000
prob_models_empirical = {k:{} for k in dct_realeccs_trimmed.keys()}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for dataset in tqdm(dct_realeccs_trimmed, total=len(dct_realeccs_trimmed)):
        #wt 1 - uniform weights
        #wt 2 - variable weights

        X = dct_realeccs_trimmed[dataset]['data']
        W_uni = np.ones_like(X)/len(X)
        W_var = dct_realeccs_trimmed[dataset]['weights']
        
        for W, wt in zip([W_uni, W_var], ['Uniform', 'Variable']):

            #Normal
            pe = 'Normal'
            pewt = f'{pe}, {wt}'
            mean = np.sum(X*W)/np.sum(W)
            std = weighted_std(X, W)
            func = norm(loc=mean, scale=std)
            prob_models_empirical[dataset][pewt] = func

            #Lognormal
            pe = 'Lognormal'
            pewt = f'{pe}, {wt}'
            shape, loc, scale = weighted_lognorm_fit(X+logfit_offset, weights=W, method='MLE')
            func = lognorm(s=shape, loc=loc-logfit_offset, scale=scale)
            prob_models_empirical[dataset][pewt] = func

            #KDE
            pe = 'KDE'
            pewt = f'{pe}, {wt}'
            bw = weighted_bw(X, W, bw_method='scott')
            func = gaussian_kde(X, bw_method=1.0, weights=W)
            bw_base = (func.covariance**0.5)[0][0]
            func.set_bandwidth(bw/bw_base)
            prob_models_empirical[dataset][pewt] = func

In [ ]:
# calculate KS-test, W-1, and W-2 between each probabilistic model and the weighted data
# ks_results = {pewt: [] for pewt in PEWT}
wass1_results_empirical = {pewt: [] for pewt in PEWT}
# wass2_results = {pewt: [] for pewt in PEWT}
for dataset in tqdm(dct_realeccs_trimmed, total=len(dct_realeccs_trimmed)):
    for pewt in PEWT:
        # get empirical data
        data = dct_realeccs_trimmed[dataset]['data']
        weights = dct_realeccs_trimmed[dataset]['weights']

        # create model PDF
        func = prob_models_empirical[dataset][pewt]
        std = np.max([np.std(data), weighted_std(data, weights)])
        lo = 0
        hi = np.max(data) + 10*std
        xplot = np.linspace(lo, hi, 1_000)
        ypdf = func.pdf(xplot)

        # # KS test
        # ycdf = cumulative_trapezoid(ypdf, xplot, initial=0)
        # ycdf = ycdf / np.max(ycdf)
        # x, y, funcecdf = weighted_ecdf(data, weights)
        # yecdf = funcecdf(xplot)
        # ksval = np.max(np.abs(ycdf-yecdf))
        # ks_results[pewt].append(ksval)

        # wass1
        wass1 = wasserstein1_weighted(data, xplot, weights, ypdf, unitless=False)
        wass1_results_empirical[pewt].append(wass1)

        # # wass2
        # wass2 = wasserstein2_weighted(data, xplot, weights, ypdf, unitless=False)
        # wass2_results[pewt].append(wass2)



# # KS
# df_ks = pd.DataFrame(ks_results, index=df_metrics.index)
# df_ks_rank = df_ks.rank(axis=1)
# df_ks_relative = df_ks.div(df_ks.mean(axis=1), axis=0)

# wass1
df_wass1_empirical = pd.DataFrame(wass1_results_empirical, index=dct_realeccs_trimmed.keys())
# df_wass1_rank = df_wass1.rank(axis=1)
# df_wass1_relative = df_wass1.div(df_wass1.mean(axis=1), axis=0)

# # wass2
# df_wass2 = pd.DataFrame(wass2_results, index=df_metrics.index)
# df_wass2_rank = df_wass2.rank(axis=1)
# df_wass2_relative = df_wass2.div(df_wass2.mean(axis=1), axis=0)




In [ ]:
# put wass scores with statistical metrics
pd.concat([df_wass1_empirical, df_realeccmetrics], axis=1).to_excel('../outputs/tables/TABLE_EmpiricalECCMetricsAndW1.xlsx')

print(df_wass1_empirical.rank(axis=1).mean())

# Comparing UQ fits

## Visualizing PDF and CDF of one example

In [ ]:
# visualize different fits
srs = df_metrics['n']
dataset = srs.loc[srs>95].loc[srs<105].index[0]
X = DATA[dataset]['data']
W = DATA[dataset]['weights']
std = weighted_std(X, W)
lo = np.max([np.min(X)-std, 0])
hi = np.max(X)+std
xplot = np.linspace(lo, hi, 1000)

fig, axes = plt.subplots(2,1, figsize=(10,4), sharex=True)
#################
# PLOT EACH PDF #
#################
ax = axes[0]
ax.plot([lo,lo],[0,0], color='black', label='Dataset')
for pewt in PEWT:
    func = prob_models[dataset][pewt]
    yplot = func.pdf(xplot)
    ax.plot(xplot, yplot, label=pewt, color=dct_colors[pewt])

# add in data points
ax.scatter(X, [0]*len(X), s=W*100_000, color='black', marker='|')

# format
ax.legend(bbox_to_anchor=(1.1,0), loc='lower right')
ax.text(0.025, 0.975, '(a) ', ha='right', va='top', fontsize=12, transform=ax.transAxes, weight='bold')
ax.text(0.025, 0.975, 'PDFs', ha='left', va='top', fontsize=12, transform=ax.transAxes)
ax.text(0.025, 0.800, dataset, ha='left', va='top', fontsize=12, transform=ax.transAxes)
# ax.set_xlim(0,)
ax.set_ylim(0,)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)


#################
# PLOT EACH CDF #
#################
ax = axes[1]
for pewt in PEWT:
    func = prob_models[dataset][pewt]
    ypdf = func.pdf(xplot)
    ycdf = scipy.integrate.cumulative_trapezoid(ypdf, xplot, initial=0)
    ycdf = ycdf / np.max(ycdf)
    ax.plot(xplot, ycdf, label=pewt, color=dct_colors[pewt])

# add in empirical CDF
x, y, func = weighted_ecdf(X, W)
yecdf = func(xplot)
yecdf = yecdf - np.min(yecdf)
yecdf = yecdf / np.max(yecdf)
ax.plot(xplot, yecdf, color='black')

# format
ax.text(0.025, 0.975, '(b) ', ha='right', va='top', fontsize=12, transform=ax.transAxes, weight='bold')
ax.text(0.025, 0.975, 'CDFs', ha='left', va='top', fontsize=12, transform=ax.transAxes)
# ax.set_xlim(0,)
ax.set_ylim(0,1.01)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
# ax.set_xlabel('ECC (kgCO2e/unit)')



plt.savefig('../outputs/figures/CompareUQMethods_FIG_PDFandCDFofUQMethods.png', bbox_inches='tight')

## Find and compare distance between UQ fits and the data

In [ ]:
# get empirical data
data = DATA[dataset]['data']
weights = DATA[dataset]['weights']

# create model PDF
func = prob_models[dataset][pewt]
std = np.max([np.std(data), weighted_std(data, weights)])
lo = 0
hi = np.max(data) + 10*std
xplot = np.linspace(lo, hi, 1_000)
ypdf = func.pdf(xplot)

# ks test
ycdf = cumulative_trapezoid(ypdf, xplot, initial=0)
ycdf = ycdf / np.max(ycdf)
x, y, funcecdf = weighted_ecdf(data, weights)
yecdf = funcecdf(xplot)
ksval = np.max(np.abs(ycdf-yecdf))

In [ ]:
# calculate KS-test, W-1, and W-2 between each probabilistic model and the weighted data
ks_results = {pewt: [] for pewt in PEWT}
wass1_results = {pewt: [] for pewt in PEWT}
wass2_results = {pewt: [] for pewt in PEWT}
for dataset in tqdm(DATA, total=len(DATA)):
    for pewt in PEWT:
        # get empirical data
        data = DATA[dataset]['data']
        weights = DATA[dataset]['weights']

        # create model PDF
        func = prob_models[dataset][pewt]
        std = np.max([np.std(data), weighted_std(data, weights)])
        lo = 0
        hi = np.max(data) + 10*std
        xplot = np.linspace(lo, hi, 1_000)
        ypdf = func.pdf(xplot)

        # KS test
        ycdf = cumulative_trapezoid(ypdf, xplot, initial=0)
        ycdf = ycdf / np.max(ycdf)
        x, y, funcecdf = weighted_ecdf(data, weights)
        yecdf = funcecdf(xplot)
        ksval = np.max(np.abs(ycdf-yecdf))
        ks_results[pewt].append(ksval)

        # wass1
        wass1 = wasserstein1_weighted(data, xplot, weights, ypdf, unitless=False)
        wass1_results[pewt].append(wass1)

        # wass2
        wass2 = wasserstein2_weighted(data, xplot, weights, ypdf, unitless=False)
        wass2_results[pewt].append(wass2)



# KS
df_ks = pd.DataFrame(ks_results, index=df_metrics.index)
df_ks_rank = df_ks.rank(axis=1)
df_ks_relative = df_ks.div(df_ks.mean(axis=1), axis=0)

# wass1
df_wass1 = pd.DataFrame(wass1_results, index=df_metrics.index)
df_wass1_rank = df_wass1.rank(axis=1)
df_wass1_relative = df_wass1.div(df_wass1.mean(axis=1), axis=0)

# wass2
df_wass2 = pd.DataFrame(wass2_results, index=df_metrics.index)
df_wass2_rank = df_wass2.rank(axis=1)
df_wass2_relative = df_wass2.div(df_wass2.mean(axis=1), axis=0)


pd.concat([df_wass1, df_metrics], axis=1).to_excel('../outputs/tables/TABLE_SyntheticECCMetricsAndW1.xlsx')

In [ ]:
# # plot PDFs
# dataset = 'dataset5711'

# data = DATA[dataset]['data']
# weights = DATA[dataset]['weights']
# xplot = np.linspace(0, np.max(data)*2, 1000)
# fig, ax = plt.subplots()


# for pewt in PEWT:
#     func = prob_models[dataset][pewt]
#     ypdf = func.pdf(xplot)
#     ycdf = scipy.integrate.cumulative_trapezoid(ypdf, xplot, initial=0)
#     ax.plot(xplot, ypdf, color=dct_colors[pewt], label=pewt)

# ax.set_xlim(0,)
# ax.set_ylim(0)
# ax.scatter(data, [0]*len(data), s=weights/np.max(weights)*500, marker='|', color='black', label='data')

In [ ]:
# # plot CDFs 

# data = DATA[dataset]['data']
# weights = DATA[dataset]['weights']
# xplot = np.linspace(0, np.max(data)*2, 1000)
# fig, ax = plt.subplots()


# for pewt in PEWT:
#     func = prob_models[dataset][pewt]
#     ypdf = func.pdf(xplot)
#     ycdf = scipy.integrate.cumulative_trapezoid(ypdf, xplot, initial=0)
#     ycdf = ycdf - np.min(ycdf)
#     ycdf = ycdf / np.max(ycdf)
#     ax.plot(xplot, ycdf, color=dct_colors[pewt], label=pewt)

# ax.set_xlim(0,)
# ax.set_ylim(0,1)

# func_data = weighted_ecdf(data, weights)[2]
# ycdf_data = func_data(xplot)
# ycdf_data = ycdf_data-np.min(ycdf_data)
# ycdf_data = ycdf_data / np.max(ycdf_data)
# ax.plot(xplot, ycdf_data, color='black', label='data')

In [ ]:
# plot wass values and rankings for each method
avg_dist = df_ks.mean().sort_values()

fig, axes = plt.subplots(len(wass1_results),1, sharex=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i, (pewt, dist) in enumerate(avg_dist.items()):
        ax = axes[i]
        dft = pd.DataFrame()
        dft['wass'] = df_ks[pewt]
        dft['rank'] = df_ks.rank(axis=1)[pewt]
        sns.stripplot(data=dft, x='wass', y=None, hue='rank', palette='viridis', ax=ax, orient='h', s=1, jitter=0.4)
        ax.get_yaxis().set_visible(False)
        ax.text(0, 0.5, f"{pewt} ({'{:.2f}'.format(np.round(dist,2))})", ha='right', va='center', transform=ax.transAxes)
        ax.spines[['top', 'right', 'left']].set_visible(False)
        
        # ax.set_xscale('log')
        
        if i != 0:
            ax.legend().set_visible(False)

ax = axes[0]
ax.text(0, 1.2, 'KS test statistic and UQ fit ranks', fontsize=12, ha='left', va='bottom', transform=ax.transAxes)
leg = ax.legend(bbox_to_anchor=(1,1), loc='upper left')
for i in range(len(leg.get_texts())):
    text = rankth[i+1]
    leg.get_texts()[i].set_text(text)

ax = axes[-1]
ax.set_xlabel('KS test statistic')

plt.savefig('../outputs/figures/CompareUQMethods_SUPP_KSTestStripAndRank.png', bbox_inches='tight')

In [ ]:
# plot wass values and rankings for each method
avg_dist = df_wass1.mean().sort_values()

fig, axes = plt.subplots(len(wass1_results),1, sharex=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i, (pewt, dist) in enumerate(avg_dist.items()):
        ax = axes[i]
        dft = pd.DataFrame()
        dft['wass'] = df_wass1[pewt]
        dft['rank'] = df_wass1.rank(axis=1)[pewt]
        sns.stripplot(data=dft, x='wass', y=None, hue='rank', palette='viridis', ax=ax, orient='h', s=1, jitter=0.4)
        ax.get_yaxis().set_visible(False)
        ax.text(0, 0.5, f"{pewt} ({'{:.2f}'.format(np.round(dist,2))})", ha='right', va='center', transform=ax.transAxes)
        ax.spines[['top', 'right', 'left']].set_visible(False)
        
        # ax.set_xscale('log')
        
        if i != 0:
            ax.legend().set_visible(False)

ax = axes[0]
ax.text(0, 1.2, 'Wasserstein-1 distance and UQ fit ranks', fontsize=12, ha='left', va='bottom', transform=ax.transAxes)
leg = ax.legend(bbox_to_anchor=(1,1), loc='upper left')
for i in range(len(leg.get_texts())):
    text = rankth[i+1]
    leg.get_texts()[i].set_text(text)

ax = axes[-1]
ax.set_xlabel('Wasserstein-1 distance')

plt.savefig('../outputs/figures/CompareUQMethods_FIG_Wass1DistStripAndRank.png', bbox_inches='tight')

In [ ]:
# plot wass values and rankings for each method
avg_dist = df_wass2.mean().sort_values()

fig, axes = plt.subplots(len(wass1_results),1, sharex=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i, (pewt, dist) in enumerate(avg_dist.items()):
        ax = axes[i]
        dft = pd.DataFrame()
        dft['wass'] = df_wass2[pewt]
        dft['rank'] = df_wass2.rank(axis=1)[pewt]
        sns.stripplot(data=dft, x='wass', y=None, hue='rank', palette='viridis', ax=ax, orient='h', s=1, jitter=0.4)
        ax.get_yaxis().set_visible(False)
        ax.text(0, 0.5, f"{pewt} ({'{:.2f}'.format(np.round(dist,2))})", ha='right', va='center', transform=ax.transAxes)
        ax.spines[['top', 'right', 'left']].set_visible(False)
        
        # ax.set_xscale('log')
        
        if i != 0:
            ax.legend().set_visible(False)

ax = axes[0]
ax.text(0, 1.2, 'Wasserstein-2 distance and UQ fit ranks', fontsize=12, ha='left', va='bottom', transform=ax.transAxes)
leg = ax.legend(bbox_to_anchor=(1,1), loc='upper left')
for i in range(len(leg.get_texts())):
    text = rankth[i+1]
    leg.get_texts()[i].set_text(text)

ax = axes[-1]
ax.set_xlabel('Wasserstein-2 distance')

plt.savefig('../outputs/figures/CompareUQMethods_SUPP_Wass2DistStripAndRank.png', bbox_inches='tight')

In [ ]:
# # distribution of relative values for wass distance
# fig, ax = plt.subplots()
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     sns.kdeplot(df_wass1_relative[PEWT], ax=ax, palette='Paired')
    
# ymin, ymax = ax.get_ylim()
# ax.vlines(1.0, ymin, ymax, color='black', linewidth=1.0, linestyle='--')
# ax.spines[['top', 'right', 'left']].set_visible(False)
# ax.get_yaxis().set_visible(False)
# ax.set_xlim(0,);
# ax.set_xlabel('Normalized Wasserstein distance such that the mean is 1.0')


# plt.savefig('../outputs/figures/CompareUQMethods_KDEplot_WassNormalized.png', bbox_inches='tight')

In [ ]:
# # plot wass values and rankings for each method
# df_results = pd.DataFrame(wass1_results).rank(axis=1)
# avg_rank = df_wass1_relative.mean().sort_values()

# fig, axes = plt.subplots(len(wass1_results),1, sharex=True, figsize=(6,3))
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     for i, (pewt, rank) in enumerate(avg_rank.items()):
#         ax = axes[i]
#         dft = pd.DataFrame()
#         dft['wass'] = df_wass1_relative[pewt]
#         dft['rank'] = df_results[pewt]
#         sns.stripplot(data=dft, x='wass', y=None, hue='rank', palette='viridis', ax=ax, orient='h', s=0.5, jitter=0.4)
#         ax.get_yaxis().set_visible(False)
#         # ax.text(0, 0.5, f"{pewt} ({np.round(df_wass1_relative.mean()[pewt],2)})", ha='right', va='center', transform=ax.transAxes)
#         ax.text(0, 0.5, f"{pewt} ({'{:.2f}'.format(np.round(rank,2))})", ha='right', va='center', transform=ax.transAxes)
#         ax.spines[['top', 'right', 'left']].set_visible(False)
        
#         ymin, ymax = ax.get_ylim()
#         ax.vlines(1.0, ymin, ymax, color='black', zorder=3, linewidth=1.0)
#         # ax.set_xscale("log")
        
#         if i != 0:
#             ax.legend().set_visible(False)

# ax = axes[0]
# ax.text(0, 1.2, 'Relative Wasserstein distances (mean=1) and UQ fit ranks', fontsize=12, ha='left', va='bottom', transform=ax.transAxes)
# leg = ax.legend(bbox_to_anchor=(1,1), loc='upper left')
# for i in range(len(leg.get_texts())):
#     text = rankth[i+1]
#     leg.get_texts()[i].set_text(text)

# ax = axes[-1]
# ax.set_xlabel('Relative Wasserstein Distance');

# plt.savefig('../outputs/figures/CompareUQMethods_Stripplot_WassNormalized_Rank.png', bbox_inches='tight')


In [ ]:
# get rank frequencies of UQ methods for empirical data
df_rank1e = df_wass1_empirical.rank(axis=1)
df_rank2e = pd.DataFrame(columns=df_rank1e.columns, index=list(set(df_rank1e.values.flatten())))

for col in df_rank1e.columns:
    df_rank2e[col] = df_rank1e[col].value_counts()
df_rank2e = df_rank2e/df_rank2e.sum().iloc[0]
ordere = (df_rank2e.T*df_rank2e.index).sum(axis=1).sort_values()

ind = [ele for ele in df_rank2e.index if int(ele)==ele]
ind.sort()
df_rank2e = df_rank2e.loc[ind, ordere.index]


In [ ]:
# get rank frequencies of UQ methods for synthetic data
df_rank1 = pd.DataFrame(wass1_results).rank(axis=1)
df_rank2 = pd.DataFrame(columns=df_rank1.columns, index=list(set(df_rank1.values.flatten())))

for col in df_rank1.columns:
    df_rank2[col] = df_rank1[col].value_counts()
df_rank2 = df_rank2/df_rank2.sum().iloc[0]
order = (df_rank2.T*df_rank2.index).sum(axis=1).sort_values()

ind = [ele for ele in df_rank2.index if int(ele)==ele]
ind.sort()
df_rank2 = df_rank2.loc[ind, order.index]

fig, axes = plt.subplots(2,1,figsize=(6,6), gridspec_kw=dict(hspace=0.5))

# synthetic data
ax = axes[0]
sns.heatmap(df_rank2.T, cmap='Blues', annot=True, fmt='.0%', ax=ax, cbar=False)
ax.set_title(f'Rank frequency for {len(DATA)} synthetic datasets (Wasserstein-1 distance)')
ax.set_xticklabels(['1st', '2nd', '3rd', '4th', '5th', '6th']);
ax.set_yticklabels([f"{pewt} ({'{:.1f}'.format(np.round(val,1))})" for (pewt, val) in order.items()])
ax.text(-0.25, 1.1, '(a)', ha='right', va='bottom', weight='bold', transform=ax.transAxes)

# empirical data
ax = axes[1]
sns.heatmap(df_rank2e.T, cmap='Blues', annot=True, fmt='.0%', ax=ax, cbar=False)
ax.set_title(f'Rank frequency for {len(df_wass1_empirical)} empirical datasets (Wasserstein-1 distance)')
ax.set_xticklabels(['1st', '2nd', '3rd', '4th', '5th', '6th']);
ax.set_yticklabels([f"{pewt} ({'{:.1f}'.format(np.round(val,1))})" for (pewt, val) in ordere.items()])
ax.text(-0.25, 1.1, '(b)', ha='right', va='bottom', weight='bold', transform=ax.transAxes)

plt.savefig('../outputs/figures/CompareUQMethods_FIG_WassRankFreq_Heatmap.png', bbox_inches='tight')


In [ ]:
lst_include = []
lst_include += list(df_rank1.loc[df_rank1['KDE, Uniform'] == 1.0].index)
lst_include += list(df_rank1.loc[df_rank1['KDE, Uniform'] == 2.0].index)
lst_include += list(df_rank1.loc[df_rank1['KDE, Variable'] == 1.0].index)
lst_include += list(df_rank1.loc[df_rank1['KDE, Variable'] == 2.0].index)
lst_include = list(set(lst_include))


f'KDE was first or second in {len(lst_include)/10_000*100}% of datasets'

In [ ]:
lst_include = []
lst_include += list(df_rank1.loc[df_rank1['Normal, Uniform'] == 6.0].index)
lst_include += list(df_rank1.loc[df_rank1['Lognormal, Uniform'] == 6.0].index)
lst_include += list(df_rank1.loc[df_rank1['Normal, Uniform'] == 5.0].index)
lst_include += list(df_rank1.loc[df_rank1['Lognormal, Uniform'] == 5.0].index)
lst_include = list(set(lst_include))

f'Lognormal/Normal Uniform was 5th or 6th in {len(lst_include)/10_000*100}% of datasets'

## Explore correlations between metrics

In [ ]:
# put all metrics into a single dataframe
df_metrics = pd.DataFrame(columns=DATA[dataset]['metrics'].keys())
for dataset in tqdm(DATA, total=len(DATA)):
    df_metrics.loc[dataset] = DATA[dataset]['metrics']


In [ ]:
# put all metrics into a single dataframe
df_metrics = pd.DataFrame(columns=DATA[dataset]['metrics'].keys())
for dataset in tqdm(DATA, total=len(DATA)):
    df_metrics.loc[dataset] = DATA[dataset]['metrics']

# add wasserstein distance, rank, and relative distance to df_metrics

dct = {col:f'{col}, dist' for col in df_wass1.columns}
dct_rank = {col:f'{col}, rank' for col in df_wass1_rank.columns}
dct_rel = {col:f'{col}, rel' for col in df_wass1_relative.columns}



df_metrics = pd.concat([
    df_metrics,
    df_wass1.rename(columns=dct),
    df_wass1.mean(axis=1).to_frame(name='wass_dist_meanbypewt'),
    df_wass1.std(axis=1).to_frame(name='wass_dist_stdbypewt'),
    df_wass1_rank.rename(columns=dct_rank),
    df_wass1_relative.rename(columns=dct_rel),
], axis=1)

In [ ]:
df_corr = pd.DataFrame(columns=['metric1', 'metric2', 'pearson', 'spearman', 'kendall'])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for metric1, metric2 in combinations(df_metrics.columns, 2):
        if metric1.startswith(metric2) or metric2.startswith(metric1):
            print(metric1, metric2)
            continue
        if metric1.startswith('stdev') and metric2.startswith('var'):
            print(metric1, metric2)
            continue
        if metric2.startswith('stdev') and metric1.startswith('var'):
            print(metric1, metric2)
            continue
        pearson, pval = pearsonr(df_metrics[metric1], df_metrics[metric2])
        spearman, pval = spearmanr(df_metrics[metric1], df_metrics[metric2])
        kendall, pval = kendalltau(df_metrics[metric1], df_metrics[metric2])

        df_corr.loc[len(df_corr)] = [metric1, metric2, pearson, spearman, kendall]


In [ ]:
# look at all the closest correlations across all metrics
df_corr['corr'] = np.abs(df_corr[['pearson', 'spearman', 'kendall']]).mean(axis=1)
df_corr = df_corr.sort_values('corr', ascending=False)
df_corr[:10]

In [ ]:
# look at all correlations for a single metric
metric = 'wass_dist_meanbypewt'
df_corr[(df_corr.values == metric).any(1)][:10]

## When are different UQ methods best/worst?

In [ ]:
# # plot wasserstein distance of each UQ method vs each metric
# metrics = list(df_metrics.columns)
# metrics.sort()
# for i in ['KDE', 'Lognormal', 'Normal']:
#     metrics = [ele for ele in metrics if ele.find(i) == -1]
# metrics.remove('mean_uw')
# metrics.remove('wass_dist_meanbypewt')
# metrics.remove('wass_dist_stdbypewt')


# metrics_plot = [
#     'coeffvar',
#     'skewness',
#     'entropy',
#     'w_v_uw_wasserstein',
#     'fit_lognorm_SW',
#     'fit_norm_SW',
#     'mode_count_est',
#     'n',
# ]

# window = 501

# ncols = 4
# nrows = int(np.ceil(len(metrics_plot)/ncols))

# fig, axes = plt.subplots(nrows, ncols, figsize=(10,nrows*1.5), gridspec_kw=dict(hspace=1.0))
# row, col = (0, 0)
# ax2del = list(product(range(nrows),range(ncols)))
# dct_rollingwass_bymetric = {metric:pd.DataFrame(columns=PEWT) for metric in metrics}
# i = 0
# for metric in tqdm(metrics_plot, total=len(metrics_plot)):
#     ax = axes[row, col]
#     ax2del.remove((row, col))
#     xavg = df_metrics.sort_values(metric)[metric]
#     dct_rollingwass_bymetric[metric]['xplot'] = xavg
#     ymax = 0.
#     for pewt in PEWT:
#         ycol = f'{pewt}, dist'
#         ax.scatter(df_metrics[metric], df_metrics[ycol], color=dct_colors[pewt], s=0.1/len(DATA), zorder=0) 
#         yavg = df_metrics.sort_values(metric)[ycol].rolling(window=window, min_periods=1, center=True, closed='both').mean()
#         dct_rollingwass_bymetric[metric][pewt] = yavg
#         ymax = np.max([ymax, np.max(yavg.values)])
#         ax.plot(xavg, yavg, color=dct_colors[pewt], linewidth=1, label=pewt, zorder=3)
    
#     # add min and max lines
#     # ymax = np.max([ymax, df_metrics[ycol].max().max()])
#     # colors_max = dct_rollingwass_bymetric[metric][PEWT].T.idxmax().map(dct_colors)
#     # colors_min = dct_rollingwass_bymetric[metric][PEWT].T.idxmin().map(dct_colors)
#     # ax.scatter(xavg, [ybest]*len(xavg), color=colors_min, marker='s', s=1)
#     # ax.scatter(xavg, [ywrst]*len(xavg), color=colors_max, marker='s', s=1)

#     # format
#     ax.text(-0.05,1.15,f'({alphabet[i]})', ha='right', va='bottom', transform=ax.transAxes, fontsize=8, weight='bold')
#     i += 1
#     ax.spines[['top', 'right']].set_visible(False)
#     # ax.set_ylabel('Wasserstein Distance')
#     ax.set_title(dct_metriclabels[metric], fontsize=8)
#     ax.set_ylim(0, ymax)
#     ax.tick_params(axis='both', which='major', labelsize=6)
#     if (row, col) == (0, ncols-1):
#         ax.legend(bbox_to_anchor=(1,1), loc='upper left', markerscale=10)
    
#     # if (row, col) != (nrows-1, 0):
#     #     ax.set_yticklabels([])

    
#     if col == ncols-1:
#         row += 1
#         col = 0
#     else:
#         col += 1

# for row, col in ax2del:
#     fig.delaxes(axes[row, col])

# ax = axes[0,0]
# ax.text(0, 1.5, 'Wasserstein-1 distance vs metrics for each UQ method', fontsize=12, ha='left', va='bottom', transform=ax.transAxes);
# # xmin, xmax = ax.get_xlim()
# # ax.text(xmin+0.25, ywrst+0.025, 'Highest Wass Distance', fontsize=6, ha='left', va='bottom');
# # ax.text(xmin+0.25, ybest-0.025, 'Lowest', fontsize=6, ha='left', va='top');
# ax.set_ylabel('Wasserstein-1 Distance', fontsize=6);

# plt.savefig('../outputs/figures/CompareUQMethods_FIG_WassDistanceVsMetric.png', bbox_inches='tight')


# # highest standard deviation in the wasserstein distances at any given point along any given metric
# # this tells us where the metrics are most different

# srs = pd.Series()
# for metric in dct_rollingwass_bymetric:
#     srs[metric] = dct_rollingwass_bymetric[metric][PEWT].std(axis=1).max()
# srs.sort_values(ascending=False).dropna()

In [ ]:
# metric = 'skewness'

# nrows = 4
# ncols = 4
# i = 0
# fig, axes = plt.subplots(nrows, ncols)

# for row, col in product(range(nrows), range(ncols)):
#     ax = axes[row, col]

#     dataset = list(df_metrics[metric].sort_values(ascending=False).index)[i]
#     i += 1

#     data = DATA[dataset]['data']
#     weights = DATA[dataset]['weights']

#     ax.scatter(data, [0]*len(data), s=weights*1000, marker='|', color='black')
#     xplot = np.linspace(-0.2,1.5,1000)
#     for pewt in PEWT:
#         func = prob_models[dataset][pewt]
#         yplot = func.pdf(xplot)
#         ax.plot(xplot, yplot, label=pewt, color=dct_colors[pewt])
#     ax.get_yaxis().set_visible(False)

# axes[0,ncols-1].legend(bbox_to_anchor=(1,1), loc='upper left')

In [ ]:
# plot wasserstein distance of each UQ method vs each metric
window = 501

ncols = 5
metrics_plot = metrics
nrows = int(np.ceil(len(metrics_plot)/ncols))
# ybest, ywrst = (ymax*0.90, ymax*0.95)

fig, axes = plt.subplots(nrows, ncols, figsize=(10,nrows*1.5), gridspec_kw=dict(hspace=1.0))
row, col = (0, 0)
ax2del = list(product(range(nrows),range(ncols)))
dct_rollingwass_bymetric = {metric:pd.DataFrame(columns=PEWT) for metric in metrics}
for i, metric in tqdm(enumerate(metrics_plot), total=len(metrics_plot)):
    ax = axes[row, col]
    ax2del.remove((row, col))
    xavg = df_metrics.sort_values(metric)[metric]
    dct_rollingwass_bymetric[metric]['xplot'] = xavg
    ymax = 0.
    for pewt in PEWT:
        ycol = f'{pewt}, dist'
        ax.scatter(df_metrics[metric], df_metrics[ycol], color=dct_colors[pewt], s=0.1/len(DATA), zorder=0) 
        yavg = df_metrics.sort_values(metric)[ycol].rolling(window=window, min_periods=1, center=True, closed='both').mean()
        dct_rollingwass_bymetric[metric][pewt] = yavg
        ymax = np.max([ymax, np.max(yavg.values)])
        ax.plot(xavg, yavg, color=dct_colors[pewt], linewidth=1, label=pewt, zorder=3)
    
    # add min and max lines
    # ymax = np.max([ymax, df_metrics[ycol].max().max()])
    # colors_max = dct_rollingwass_bymetric[metric][PEWT].T.idxmax().map(dct_colors)
    # colors_min = dct_rollingwass_bymetric[metric][PEWT].T.idxmin().map(dct_colors)
    # ax.scatter(xavg, [ybest]*len(xavg), color=colors_min, marker='s', s=1)
    # ax.scatter(xavg, [ywrst]*len(xavg), color=colors_max, marker='s', s=1)

    # format
    ax.spines[['top', 'right']].set_visible(False)
    # ax.set_ylabel('Wasserstein Distance')
    ax.set_title(dct_metriclabels[metric], fontsize=7)
    ax.set_ylim(0, ymax)
    ax.tick_params(axis='both', which='major', labelsize=6)
    if i == len(metrics)-1:
        ax.legend(bbox_to_anchor=(1.5, 1.5), loc='upper left', markerscale=10, fontsize=8)
    ax.text(-0.05,1.15,f'({alphabet[i]})', ha='right', va='bottom', transform=ax.transAxes, fontsize=8, weight='bold')
    
    # if (row, col) != (nrows-1, 0):
    #     ax.set_yticklabels([])

    
    if col == ncols-1:
        row += 1
        col = 0
    else:
        col += 1

for row, col in ax2del:
    fig.delaxes(axes[row, col])

ax = axes[0,0]
ax.text(0, 1.5, 'Wasserstein-1 distance vs metrics for each UQ method', fontsize=12, ha='left', va='bottom', transform=ax.transAxes);
# xmin, xmax = ax.get_xlim()
# ax.text(xmin+0.25, ywrst+0.025, 'Highest Wass Distance', fontsize=6, ha='left', va='bottom');
# ax.text(xmin+0.25, ybest-0.025, 'Lowest', fontsize=6, ha='left', va='top');
ax.set_ylabel('Wasserstein-1 Distance', fontsize=6)

plt.savefig('../outputs/figures/CompareUQMethods_SUPP_WassDistanceVsMetric_ALLMETRICS.png', bbox_inches='tight')

# highest standard deviation in the wasserstein distances at any given point along any given metric
# this tells us where the metrics are most different

srs = pd.Series()
for metric in dct_rollingwass_bymetric:
    srs[metric] = dct_rollingwass_bymetric[metric][PEWT].std(axis=1).max()
srs.sort_values(ascending=False).dropna()

In [ ]:
srs = df_metrics['weight_outliers']
srs.loc[srs==0]